In [ ]:
# pd.read_csv(r"S:\2026\09-Sept-26\26.09.26\30978098_BEINDATANEWRPT.CSV", nrows=1000)

check CNE HEAD OFFICE trend

In [ ]:
2from pathlib import Path
import pandas as pd


# =========================
# Configuration
# =========================

ROOT_PATH = r"S:\2026\09-Sept-26"
OUTPUT_FILE = r"all_results.csv"

OUTPUT_COLUMNS = [
    "Customer Number",
    "Customer Type",
    "Entity",
]


# =========================
# Read CSV
# =========================

def read_csv_with_encoding(file_path):

    try:
        df = pd.read_csv(
            file_path,
            dtype=str,
            encoding="utf-8"
        )
    except UnicodeDecodeError:
        df = pd.read_csv(
            file_path,
            dtype=str,
            encoding="cp1256"
        )

    # Normalize column names
    df.columns = df.columns.str.strip()

    # Case-insensitive column mapping
    column_map = {
        col.upper(): col
        for col in df.columns
    }

    required_input = [
        "CUSTOMER NUMBER",
        "CUSTOMER TYPE",
        "ENTITY",
        "STATUS"
    ]

    missing = [
        col for col in required_input
        if col not in column_map
    ]

    if missing:
        raise ValueError(
            f"Missing columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    # Standardize column names
    df = df.rename(columns={
        column_map["CUSTOMER NUMBER"]: "Customer Number",
        column_map["CUSTOMER TYPE"]: "Customer Type",
        column_map["ENTITY"]: "Entity",
        column_map["STATUS"]: "Status",
    })

    return df[OUTPUT_COLUMNS]


def process_reports(root_path, output_file):

    root = Path(root_path)

    all_results = []

    for date_folder in sorted(root.iterdir()):

        if not date_folder.is_dir():
            continue

        date_value = date_folder.name

        # Find matching files
        matching_files = [
            f
            for f in date_folder.rglob("*BEINDATANEWRPT*")
            if f.is_file()
        ]

        if not matching_files:
            print(f"[SKIP] {date_value}: no matching file")
            continue

        # Take most recently created file
        selected_file = max(
            matching_files,
            key=lambda f: f.stat().st_ctime
        )

        print(f"[READ] {date_value}: {selected_file}")

        try:

            df = read_csv_with_encoding(selected_file)

            # ==========================================
            # Filter Entity
            # ==========================================

            df["Entity"] = df["Entity"].str.strip()

            df = df[
                df["Entity"] == "CNE Head office"
            ]

            # ==========================================
            # Drop duplicate customers
            # ==========================================

            df = df.drop_duplicates(
                subset=["Customer Number"],
                keep="first"
            )

            # ==========================================
            # Add date
            # ==========================================

            df["Date"] = date_value

            # Put Date first
            df = df[
                ["Date"] + OUTPUT_COLUMNS
            ]

            all_results.append(df)

            print(f"       Rows after filtering: {len(df):,}")

        except Exception as e:

            print(f"[ERROR] {selected_file}")
            print(f"        {e}")


    # ==========================================
    # Combine all dates
    # ==========================================

    if not all_results:
        print("No data was found.")
        return

    result = pd.concat(
        all_results,
        ignore_index=True
    )

    result.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n==============================")
    print(f"Total rows : {len(result):,}")
    print(f"Output     : {output_file}")
    print("==============================")
# =========================
# Run
# =========================

process_reports(
    ROOT_PATH,
    OUTPUT_FILE
)

In [12]:
df1 = pd.read_csv("all_results.csv")

In [14]:
df1['Date'] = pd.to_datetime(df1['Date'], dayfirst=True)

C:\Users\mturky\AppData\Local\Temp\1\ipykernel_28840\1453910728.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df1['Date'] = pd.to_datetime(df1['Date'], dayfirst=True)


In [19]:
# Make sure Date is actually a datetime
df1["Date"] = pd.to_datetime(df1["Date"])

# Sort by date
df1 = df1.sort_values("Date")

# For each row, check whether the customer appears on any later date
df1["exists_later"] = (
    df1.groupby("Customer Number")["Date"]
      .transform("max")
    > df1["Date"]
)

# Keep customers that don't exist in any following day
result = df1[~df1["exists_later"]].copy()

# Remove helper column
result = result.drop(columns="exists_later").to_csv(r'all_doaa2.csv', index=False)

In [18]:
result.loc[result['Date']!='2026-09-30']

,Date,Customer Number,Customer Type,Entity
589094,2026-09-02,18680761,CNE Subscriber,CNE Head office
602925,2026-09-02,18758104,beIN Quartar Installment,CNE Head office
531836,2026-09-02,16381976,beIN Bi Installment,CNE Head office
563741,2026-09-02,17281673,beIN Quartar Installment,CNE Head office
549549,2026-09-02,17331493,CNE Subscriber,CNE Head office
...,...,...,...,...
9930316,2026-09-29,18920650,CNE Subscriber,CNE Head office
9936444,2026-09-29,18903234,beIN Quartar Installment,CNE Head office
9935873,2026-09-29,18905255,beIN Quartar Installment,CNE Head office
9962715,2026-09-29,18172673,CNE Subscriber,CNE Head office
